In [1]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
from qiskit_aer import AerSimulator

In [2]:
def qft_rotations(qc, n):
    if n == 0:
        return qc
    n -= 1
    qc.h(n)

    for qubit in range(n):
        angle = np.pi / 2**(n - qubit)
        qc.cp(angle, qubit, n)

    qc.barrier()
    return qft_rotations(qc, n)

In [3]:
def swap_registers(qc, n):
    for i in range(n // 2):
        qc.swap(i, n - i - 1)
    return qc

In [4]:
def build_qft(n):
    qc = QuantumCircuit(n)
    qft_rotations(qc, n)
    swap_registers(qc, n)
    return qc

In [5]:
n = 3
qft = build_qft(n)
print(qft.draw())

                             ░                ░ ┌───┐ ░    
q_0: ──────■─────────────────░───────■────────░─┤ H ├─░──X─
           │                 ░ ┌───┐ │P(π/2)  ░ └───┘ ░  │ 
q_1: ──────┼────────■────────░─┤ H ├─■────────░───────░──┼─
     ┌───┐ │P(π/4)  │P(π/2)  ░ └───┘          ░       ░  │ 
q_2: ┤ H ├─■────────■────────░────────────────░───────░──X─
     └───┘                   ░                ░       ░    


In [6]:
circuit = QuantumCircuit(n)
circuit.x(0)
circuit.x(1)
circuit.barrier()

qft_inst = build_qft(n).to_instruction()
circuit.append(qft_inst, range(n))

state = Statevector.from_instruction(circuit)

for i, amplitude in enumerate(state):
    phase = np.angle(amplitude, deg=True)
    if phase < 0:
        phase += 360
    print(f"State |{i:03b}> ({i}): Amplitude = {np.abs(amplitude):.3f}, Phase = {phase:.1f}°")

State |000> (0): Amplitude = 0.354, Phase = 0.0°
State |001> (1): Amplitude = 0.354, Phase = 135.0°
State |010> (2): Amplitude = 0.354, Phase = 270.0°
State |011> (3): Amplitude = 0.354, Phase = 45.0°
State |100> (4): Amplitude = 0.354, Phase = 180.0°
State |101> (5): Amplitude = 0.354, Phase = 315.0°
State |110> (6): Amplitude = 0.354, Phase = 90.0°
State |111> (7): Amplitude = 0.354, Phase = 225.0°
